# Tutorial 1: Building a Small AHP Model for MSW-to-Energy Technology Ranking

This notebook introduces a small **Analytic Hierarchy Process (AHP)** model for ranking municipal solid waste-to-energy (MSW-to-energy) technology alternatives.

The notebook is organised step by step:

1. Define the AHP hierarchy structure.
2. Register criteria, sub-criteria, deciding factors, and alternatives.
3. Build pairwise comparison matrices from cardinal cost data.
4. Calculate AHP priority vectors using the eigenvector method.
5. Check consistency using Saaty's Consistency Ratio.
6. Rank alternatives for CAPEX, OPEX, LCOE, and the combined economical criterion.

The example focuses mainly on the **Economical** criterion, using CAPEX, OPEX, and LCOE data.

    

## 1. Import required libraries

Only lightweight Python libraries are used in this tutorial:

- `dataclasses` for defining the AHP node structure.
- `typing` for clearer function definitions.
- `numpy` for matrix calculations.
- `pandas` for tables and ranking outputs.
- `re` for creating stable text identifiers.
    

In [10]:
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Iterable, Tuple
import numpy as np
import pandas as pd
import re

## 2. Define the AHP node structure

Each element in the hierarchy is represented as an `AHPNode`.

A node can represent:

- the overall goal,
- a main criterion,
- a sub-criterion,
- a deciding factor,
- or an alternative.

The same data structure is used for all levels of the hierarchy, which makes the model flexible and reusable.

In [11]:
@dataclass
class AHPNode:
    """
    Generic AHP node.

    A node can represent a goal, criterion, sub-criterion,
    deciding factor, or alternative.
    """

    name: str
    kind: str  # 'goal' | 'criterion' | 'subcriterion' | 'factor' | 'alternative'

    parent: Optional["AHPNode"] = None
    children: List["AHPNode"] = field(default_factory=list)

    # Optional metadata
    direction: Optional[str] = None
    unit: Optional[str] = None

    # Placeholders for AHP calculations
    pairwise: Optional[np.ndarray] = None
    local_weights: Optional[np.ndarray] = None

    def add(self, child: "AHPNode") -> "AHPNode":
        """Add a child node and return it."""
        child.parent = self
        self.children.append(child)
        return child

    def is_leaf(self) -> bool:
        """Return True if the node has no children."""
        return len(self.children) == 0

    def path(self) -> List[str]:
        """Return the full path from root to node."""
        node = self
        parts = []

        while node is not None:
            parts.append(node.name)
            node = node.parent

        return list(reversed(parts))

## 3. Helper functions for working with the hierarchy

The following helper functions allow us to:

- print the hierarchy as a tree,
- iterate through all nodes,
- create stable identifiers,
- and export deciding factors into a table.

In [12]:
def print_tree(node: AHPNode, indent: int = 0) -> None:
    """Print the AHP hierarchy as a readable tree."""

    icon = {
        "goal": "🎯",
        "criterion": "◼︎",
        "subcriterion": "▪︎",
        "factor": "•",
        "alternative": "–",
    }.get(node.kind, "•")

    meta = []

    if node.direction:
        meta.append(f"dir={node.direction}")

    if node.unit:
        meta.append(f"unit={node.unit}")

    meta_text = f" [{' | '.join(meta)}]" if meta else ""

    print("  " * indent + f"{icon} {node.name}{meta_text}")

    for child in node.children:
        print_tree(child, indent + 1)


def iter_nodes(root: AHPNode) -> Iterable[AHPNode]:
    """Iterate through all nodes in the hierarchy."""

    stack = [root]

    while stack:
        node = stack.pop()
        yield node
        stack.extend(reversed(node.children))


def slug(text: str) -> str:
    """Create a stable identifier."""

    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", "_", text).strip("_")

    return text


def factors_table(root: AHPNode) -> pd.DataFrame:
    """Return a table of all deciding factors."""

    rows = []

    for node in iter_nodes(root):

        if node.kind == "factor":

            rows.append({
                "id": slug("/".join(node.path())),
                "name": node.name,
                "path": " > ".join(node.path()),
                "direction": node.direction,
                "unit": node.unit,
            })

    return (
        pd.DataFrame(rows)
        .sort_values("path")
        .reset_index(drop=True)
    )

## 4. Build the MSW-to-energy AHP hierarchy

The hierarchy contains four main criteria:

1. Technological
2. Economical
3. Environmental
4. Socio-cultural

The alternatives are stored separately because each deciding factor can later compare the alternatives locally.

In [13]:
def build_mswt_ahp_hierarchy() -> Dict[str, object]:

    # Goal
    root = AHPNode(
        "Rank MSW-to-Energy Technology Alternatives",
        kind="goal"
    )

    # Alternatives
    alternatives = [
        "INC-E",
        "INC-H",
        "AD",
        "PYR",
        "GAS",
        "HTC",
        "Composting",
        "INC-E-CCS 85%",
        "INC-E-CCS 95%",
        "INC-H-CCS- 85%",
        "INC-H-CCS- 95%",
    ]

    alternative_nodes = [
        AHPNode(name, kind="alternative")
        for name in alternatives
    ]

    # Main criteria
    tech = root.add(AHPNode("Technological", kind="criterion"))
    eco = root.add(AHPNode("Economical", kind="criterion"))
    env = root.add(AHPNode("Environmental", kind="criterion"))
    soc = root.add(AHPNode("Socio-cultural", kind="criterion"))

    # Technological
    feedstock = tech.add(AHPNode("Feedstock", kind="subcriterion"))

    feedstock.add(
        AHPNode(
            "Moisture Content",
            kind="factor",
            direction="min",
            unit="%"
        )
    )

    feedstock.add(
        AHPNode(
            "Calorific Value",
            kind="factor",
            direction="max",
            unit="MJ/kg"
        )
    )

    feedstock.add(
        AHPNode(
            "Pre-Processing",
            kind="factor",
            direction="min"
        )
    )

    # Economical
    capex_node = eco.add(
        AHPNode(
            "CAPEX",
            kind="factor",
            direction="min",
            unit="EUR/Ton"
        )
    )

    opex_node = eco.add(
        AHPNode(
            "OPEX",
            kind="factor",
            direction="min",
            unit="EUR/Ton"
        )
    )

    lcoe_node = eco.add(
        AHPNode(
            "LCOE",
            kind="factor",
            direction="min",
            unit="EUR/MJ"
        )
    )

    return {
        "root": root,
        "alternatives": alternative_nodes,
        "eco": eco,
        "capex_node": capex_node,
        "opex_node": opex_node,
        "lcoe_node": lcoe_node,
    }

## 5. Build and preview the hierarchy

In [14]:
ahp = build_mswt_ahp_hierarchy()

root = ahp["root"]
alternatives = ahp["alternatives"]

print("AHP hierarchy:\n")

print_tree(root)

print("\nAlternatives:")

for alt in alternatives:
    print(f"  - {alt.name}")

AHP hierarchy:

🎯 Rank MSW-to-Energy Technology Alternatives
  ◼︎ Technological
    ▪︎ Feedstock
      • Moisture Content [dir=min | unit=%]
      • Calorific Value [dir=max | unit=MJ/kg]
      • Pre-Processing [dir=min]
  ◼︎ Economical
    • CAPEX [dir=min | unit=EUR/Ton]
    • OPEX [dir=min | unit=EUR/Ton]
    • LCOE [dir=min | unit=EUR/MJ]
  ◼︎ Environmental
  ◼︎ Socio-cultural

Alternatives:
  - INC-E
  - INC-H
  - AD
  - PYR
  - GAS
  - HTC
  - Composting
  - INC-E-CCS 85%
  - INC-E-CCS 95%
  - INC-H-CCS- 85%
  - INC-H-CCS- 95%
